In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.utils import resample
from sklearn.metrics import classification_report, accuracy_score


In [22]:
# 1. Load Data
data = pd.read_csv('D:\KULIAHHH\clustercc\data\CCDATA.csv')  # Ganti dengan path dataset Anda
data

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.40,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.00,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.00,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.00,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.00,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8945,C19186,28.493517,1.000000,291.12,0.00,291.12,0.000000,1.000000,0.000000,0.833333,0.000000,0,6,1000.0,325.594462,48.886365,0.500000,6
8946,C19187,19.183215,1.000000,300.00,0.00,300.00,0.000000,1.000000,0.000000,0.833333,0.000000,0,6,1000.0,275.861322,NaN,0.000000,6
8947,C19188,23.398673,0.833333,144.40,0.00,144.40,0.000000,0.833333,0.000000,0.666667,0.000000,0,5,1000.0,81.270775,82.418369,0.250000,6
8948,C19189,13.457564,0.833333,0.00,0.00,0.00,36.558778,0.000000,0.000000,0.000000,0.166667,2,0,500.0,52.549959,55.755628,0.250000,6


In [23]:
# 2. Rename Kolom
data = data.rename(columns={
    'CUST_ID': 'ID_Pelanggan',
    'BALANCE': 'Saldo',
    'BALANCE_FREQUENCY': 'Frekuensi_Saldo',
    'PURCHASES': 'Pembelian',
    'ONEOFF_PURCHASES': 'Pembelian_Sekali',
    'INSTALLMENTS_PURCHASES': 'Pembelian_Cicilan',
    'CASH_ADVANCE': 'Uang_Muka',
    'PURCHASES_FREQUENCY': 'Frekuensi_Pembelian',
    'ONEOFF_PURCHASES_FREQUENCY': 'Frekuensi_Pembelian_Sekali',
    'PURCHASES_INSTALLMENTS_FREQUENCY': 'Frekuensi_Pembelian_Cicilan',
    'CASH_ADVANCE_FREQUENCY': 'Frekuensi_Uang_Muka',
    'CASH_ADVANCE_TRX': 'Transaksi_Uang_Muka',
    'PURCHASES_TRX': 'Transaksi_Pembelian',
    'CREDIT_LIMIT': 'Batas_Kredit',
    'PAYMENTS': 'Pembayaran',
    'MINIMUM_PAYMENTS': 'Pembayaran_Minimum',
    'PRC_FULL_PAYMENT': 'Pembayaran_Penuh',
    'TENURE': 'Tenur'
})

# Tampilkan beberapa baris data setelah rename untuk memastikan kolom telah berhasil diganti
print(data.head())

  ID_Pelanggan        Saldo  Frekuensi_Saldo  Pembelian  Pembelian_Sekali  \
0       C10001    40.900749         0.818182      95.40              0.00   
1       C10002  3202.467416         0.909091       0.00              0.00   
2       C10003  2495.148862         1.000000     773.17            773.17   
3       C10004  1666.670542         0.636364    1499.00           1499.00   
4       C10005   817.714335         1.000000      16.00             16.00   

   Pembelian_Cicilan    Uang_Muka  Frekuensi_Pembelian  \
0               95.4     0.000000             0.166667   
1                0.0  6442.945483             0.000000   
2                0.0     0.000000             1.000000   
3                0.0   205.788017             0.083333   
4                0.0     0.000000             0.083333   

   Frekuensi_Pembelian_Sekali  Frekuensi_Pembelian_Cicilan  \
0                    0.000000                     0.083333   
1                    0.000000                     0.000000   
2 

In [24]:
print("Data Shape :" ,data.shape)
data.info()

Data Shape : (8950, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8950 entries, 0 to 8949
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID_Pelanggan                 8950 non-null   object 
 1   Saldo                        8950 non-null   float64
 2   Frekuensi_Saldo              8950 non-null   float64
 3   Pembelian                    8950 non-null   float64
 4   Pembelian_Sekali             8950 non-null   float64
 5   Pembelian_Cicilan            8950 non-null   float64
 6   Uang_Muka                    8950 non-null   float64
 7   Frekuensi_Pembelian          8950 non-null   float64
 8   Frekuensi_Pembelian_Sekali   8950 non-null   float64
 9   Frekuensi_Pembelian_Cicilan  8950 non-null   float64
 10  Frekuensi_Uang_Muka          8950 non-null   float64
 11  Transaksi_Uang_Muka          8950 non-null   int64  
 12  Transaksi_Pembelian          8950 non-null   int64  

In [25]:
# Memilih fitur yang relevan untuk klasifikasi
X = data[['Saldo', 'Frekuensi_Saldo', 'Pembelian', 'Pembelian_Sekali',
          'Pembelian_Cicilan', 'Uang_Muka', 'Frekuensi_Pembelian',
          'Frekuensi_Pembelian_Sekali', 'Frekuensi_Pembelian_Cicilan',
          'Frekuensi_Uang_Muka', 'Batas_Kredit', 'Pembayaran',
          'Pembayaran_Minimum', 'Pembayaran_Penuh', 'Tenur']]

In [26]:
# 2. Fitur dan Target
X = data.drop(['Pembayaran_Penuh', 'Pembayaran_Minimum'], axis=1)
y = (data['Pembayaran_Penuh'] > data['Pembayaran_Minimum']).astype(int)


In [27]:
# 3. Menangani Missing Values
X = X.fillna(0)
y = y.dropna()
X = X.loc[y.index]  # Hanya gunakan data dengan target yang valid


In [28]:
# 4. Cek Distribusi Target
print("Distribusi Kelas Sebelum Resampling:")
print(y.value_counts())

Distribusi Kelas Sebelum Resampling:
0    8950
Name: count, dtype: int64


In [ ]:
if y.nunique() == 1:
    print("Target hanya memiliki satu kelas.")
else:
    # Oversampling
    X['target'] = y
    class_0 = X[X['target'] == 0]
    class_1 = X[X['target'] == 1]

    class_1_over = resample(class_1,
                            replace=True,
                            n_samples=len(class_0),
                            random_state=42)

    data_balanced = pd.concat([class_0, class_1_over])
    X = data_balanced.drop('target', axis=1)
    y = data_balanced['target']

    print("Distribusi Kelas Setelah Resampling:")
    print(y.value_counts())

Target hanya memiliki satu kelas.


In [31]:
# 7. Normalisasi Data
X = X.apply(pd.to_numeric, errors='coerce')  # Konversi semua ke numerik, non-numerik jadi NaN
X = X.fillna(0)  # Ganti NaN dengan 0 setelah konversi
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [32]:
# 7. Pembagian Data Training dan Testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)


In [34]:
# 8. Model 1: SVM (Model Utama)
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)
svm_pred = svm_model.predict(X_test)

print("\nEvaluasi Model SVM:")
print(classification_report(y_test, svm_pred))
print("Akurasi:", accuracy_score(y_test, svm_pred))

ValueError: The number of classes has to be greater than one; got 1 class

In [ ]:
# 9. Model 2: Decision Tree (Model Pembanding)
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

print("\nEvaluasi Model Decision Tree:")
print(classification_report(y_test, dt_pred))
print("Akurasi:", accuracy_score(y_test, dt_pred))

In [ ]:
# 10. Segmentasi: KMeans dan GMM
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans_labels = kmeans.fit_predict(X_scaled)

print("\nHasil Cluster KMeans:")
print(pd.Series(kmeans_labels).value_counts())

gmm = GaussianMixture(n_components=2, random_state=42)
gmm_labels = gmm.fit_predict(X_scaled)

print("\nHasil Cluster GMM:")
print(pd.Series(gmm_labels).value_counts())
